In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
from tqdm import tqdm    # Shows progress bar
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import seaborn as sns

import os

from PIL import Image

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from torchvision import models
import torch.nn as nn
import torch

import kagglehub
import os
import glob

import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

from PIL import Image
from torchvision.datasets import ImageFolder

from sklearn.model_selection import train_test_split



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps values to 0, 1, 2... based on unique values present

    # Converts mask to LongTensor                                               ; Segmentation masks must be LongTensor
    mask = mask.long()

    # Extracts all unique pixel values in the mask
    unique_values = torch.unique(mask)

    # Creates a new empty mask SAME shape as original mask
    remapped_mask = torch.zeros_like(mask)


    # Sorts the original values  +  Assigns new indices starting from 0
    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val


    # Returns the cleaned, class-indexed mask
    return remapped_mask



# Custom DS Class
class MulticlassDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, mask_transform=None):
        #self.image_paths = image_paths
        #self.mask_paths = mask_paths
        # 1️⃣ Get ALL image paths from Folder
        self.image_paths =  sorted(glob.glob(f"{root_dir}/images/*.jpg"))
        self.mask_paths =  sorted(glob.glob(f"{root_dir}/masks/*.png")) # Check extension!
        df = len(self.image_paths) + len(self.mask_paths)

        # Image transformation
        self.transform = transform

        # Mask transformations (PILToTensor only, no normalization)
        self.mask_transform = mask_transform



    #------------------------------------------------------------------__len__()
    def __len__(self):
        # Total number of images
        return len(self.image_paths)


    #----------------------------------------------------------------_getitem_()
    def __getitem__(self, idx):
        # 1. Load Image & Mask
        image = Image.open(self.image_paths[idx]).convert("RGB")                # 3-channel image
        mask = Image.open(self.mask_paths[idx]).convert("L")                    # 1-channel Mask ; Grayscale for masks ; Keep as L (grayscale)


        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        if self.mask_transform:
            mask = self.mask_transform(mask)


        # 3. Remap Mask (we need this because CrossEntropy requires the labels to be consecutive)
        mask = remap_mask(mask)

        return image, mask



In [ ]:
# data is not splitted here, we will use only the training folder, then split it.
# 1. Define Paths
root_dir = os.path.join(path, "dataset")
all_images = sorted(glob.glob(f"{root_dir}/images/*.jpg"))
all_masks  = sorted(glob.glob(f"{root_dir}/masks/*.png")) # Check extension!

print(path)
print(root_dir)
print(all_images)
print(all_masks)


# 2. Split Data
train_imgs, test_imgs, train_masks, test_masks = train_test_split(
    all_images,
    all_masks,
    test_size=0.2,
    random_state=42
)

print(f"Total: {len(train_imgs+train_masks)}, Train: {len(train_imgs)}, Test: {len(test_imgs)}")



# 3. Define Transform
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()         # ToTensor does two things: Convert to tensor + scaling (divide by 255)
])


transform_mask = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.PILToTensor()       # PILToTensor does one thing: Convert to tensor only (mask should not be scaled!!)
])



# 4. Create Datasets
'''
train_dataset = MulticlassDataset(
    train_imgs,
    train_masks,
    transform=transform,
    mask_transform=transform_mask
    )

test_dataset  = MulticlassDataset(
    test_imgs,
    test_masks,
    transform=transform,
    mask_transform=transform_mask
    )
'''
train_dataset = MulticlassDataset(
    os.path.join(path, "dataset"),
    os.path.join(path, "dataset"),
    transform=transform,
    mask_transform=transform_mask
    )

test_dataset = MulticlassDataset(
    os.path.join(path, "dataset"),
    os.path.join(path, "dataset"),
    transform=transform,
    mask_transform=transform_mask
    )

# 5. Check Output
img, mask = train_dataset[0]
print(f"Img Shape: {img.shape}")   # [3, 256, 256]
print(f"Mask Shape: {mask.shape}") # [1, 256, 256]
print(f"Unique Classes: {torch.unique(mask)}")

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


model = smp.Unet(
  encoder_name="efficientnet-b1",
  encoder_weights="imagenet",
  in_channels=3,  # cuz RGB
  classes=8,      # cuz Class Nums
).to(device)

model = model.to(device)

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,                            # Number of images per batch
    shuffle=True                              # Randomizes training data → better learning
    )

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False                             # Keeps test data order fixed for evaluation
    )


print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")


In [ ]:
# TO DO
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  total_loss = 0

  for images, masks in tqdm(dataloader):
    # Move data to device
    images, masks = images.to(device), masks.to(device).float()

    #Complete the training step
    # 1. Forward pass
    # 2. Compute loss
    # 3. Zero gradients
    # 4. Backward pass
    # 5. Update weights
    outputs = model(images)
    loss = criterion(outputs, masks)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
# validation function

def validate(model, dataloader, criterion, device):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for images, masks in dataloader:
      images, masks = images.to(device), masks.to(device).float()

      # Complete the validation step
      # 1. Forward pass
      # 2. Compute loss
      outputs = model(images)
      loss = criterion(outputs, masks)

      total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
# TO DO
from torch import nn

# YOUR CODE HERE
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Train for 5 epochs
num_epochs = 10

In [ ]:
# Run training
train_losses = []
val_losses = []

for epoch in range(num_epochs):
  train_loss = train_one_epoch(
      model, train_loader, criterion, optimizer, device)
  val_loss = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
# Visualize predictions on test images
import random

model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get random test samples
indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(indices):
  image, mask = test_dataset[idx]

  # TODO: Get model prediction
  # 1. Add batch dimension: image.unsqueeze(0)
  # 2. Move to device
  # 3. Get prediction: model(image)
  # 4. Apply sigmoid to get probabilities
  # 5. Threshold at 0.5 to get binary mask

  with torch.no_grad():
    input_tensor = image.unsqueeze(0).to(device)
    output = model(input_tensor)
    pred = torch.sigmoid(output)
    pred = (pred > 0.5).float().cpu()

  # Display results
  axes[i, 0].imshow(denormalize(image))
  axes[i, 0].set_title("MRI Image")
  axes[i, 0].axis("off")

  axes[i, 1].imshow(mask.squeeze(), cmap="gray")
  axes[i, 1].set_title("Ground Truth")
  axes[i, 1].axis("off")

  axes[i, 2].imshow(pred.squeeze(), cmap="gray")
  axes[i, 2].set_title("Prediction")
  axes[i, 2].axis("off")

plt.tight_layout()
plt.show()